In [58]:
import mne
from sklearn.metrics import accuracy_score
import os
from mne import Epochs, events_from_annotations, pick_types
import numpy as np
import joblib
import pandas as pd
import ast

random_state=42
mne.set_log_level('CRITICAL')

In [15]:
colors = {
    'AW': '#ef8a62',
    'MO': '#67a9cf',
    'MI': '#999999',
    'AW_2': '#613828'
}

#data_folder = "raw_data"
data_folder = "raw_data"
ERP_channels =  ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'Oz'] # Updated channel list
FBCSP_channels = ['C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8'] # Updated channel list

In [16]:
def AW_annotations_online(raw, AW_labels):
    #new_annotations = raw.annotations.description
    new_annotations=[]
    for annotation in raw.annotations:
        if '%' not in annotation['description']:
            new_annotations.append(annotation)
    for j in new_annotations:
        if j['description']=='Start_cue_Arm':
            filename = 'conditions_male.xlsx'
            break
        if j['description']=='Start_cue_Leg':
            filename = 'conditions_female.xlsx'
            break   
    df_condition = pd.read_excel(filename)
    n=0
    for j in new_annotations:
        if j['description'] =='Start_cue_Arm' or j['description']=='Start_cue_Leg':
            if df_condition.loc[n, 'Cue'] == 'AW':
                if AW_labels == 'New_labels':
                    if df_condition.loc[n, 'Visual_Cue']=='APLAUDIR':
                        j['description']='AW_APLAUDIR'
                    elif df_condition.loc[n, 'Visual_Cue']=='PATEAR':
                        j['description']='AW_PATEAR'
                    elif df_condition.loc[n,'Condition']=='Leg':
                        j['description']='AW_Leg'
                    elif df_condition.loc[n,'Condition']=='Arm':
                        j['description']='AW_Arm'
                else:
                    j['description']='_'.join(df_condition.values[n][1:-1])
            else:
                j['description']='_'.join(df_condition.values[n][1:-1])
            n=n+1
    
    annotations = mne.Annotations(
        onset=[a['onset'] for a in new_annotations],
        duration=[a['duration'] for a in new_annotations],
        description=[a['description'] for a in new_annotations]
    )
    raw.set_annotations(annotations)     

In [17]:
def create_online_epochs(filename, condition, aw_labels, channels, tmin_tmax, baseline, clean_data = True):
   
   raw = mne.io.read_raw_eeglab(filename, preload=True)
   #
   if clean_data:
      raw.filter(l_freq=1.0, h_freq=None,  verbose=False)
      raw.filter(l_freq=None, h_freq=40.0,  verbose=False)
      
   picks = pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)
    #Start_cue_leg_TRETEN  Start_cue_Arm_KLATSCHEN
   AW_annotations_online(raw, aw_labels)
   events, events_id = events_from_annotations(raw, verbose=False)

   if condition == 'AW':
      events, events_id = events_from_annotations(raw, verbose=False)
      if aw_labels == 'New_labels':
         event_id= {'AW_Arm': events_id['AW_Arm'], 'AW_Leg': events_id['AW_Leg'], 
                   'AW_APLAUDIR': events_id['AW_APLAUDIR'], 'AW_PATEAR': events_id['AW_PATEAR']}
      else:
         event_id= {'AW_Arm': events_id['AW_Arm'], 'AW_Leg': events_id['AW_Leg']}
   elif condition == 'MI':
      event_id={'MI_Arm': events_id['MI_Arm'], 'MI_Leg': events_id['MI_Leg']}
   elif condition == 'MO':
      event_id={'MO_Arm': events_id['MO_Arm'], 'MO_Leg': events_id['MO_Leg']}

        
   epochs = Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=tmin_tmax[0],
    tmax=tmin_tmax[1],
    proj=True,
    picks=picks,
    baseline=baseline,
    preload=True,
    detrend=0)

   return epochs.pick_channels(channels)   

In [18]:
def align_epochs_and_labels(fbcsp_epochs, erp_epochs, arm_label='Start_cue_Arm', leg_label='Start_cue_Leg'):

    fbcsp_events = fbcsp_epochs.events
    erp_events = erp_epochs.events

            # Find common events
    common_event_ids_fbcsp = set(tuple(row) for row in fbcsp_events)
    common_event_ids_erp = set(tuple(row) for row in erp_events)
    common_events = list(common_event_ids_fbcsp.intersection(common_event_ids_erp))

            # Get indices for alignment
    fbcsp_indices = [np.where((fbcsp_events == common_event).all(axis=1))[0][0] for common_event in common_events]
    erp_indices = [np.where((erp_events == common_event).all(axis=1))[0][0] for common_event in common_events]

            # Align epochs
    fbcsp_aligned = fbcsp_epochs[fbcsp_indices]
    erp_aligned = erp_epochs[erp_indices]

            # Create label mapping and labels
    mapping_offline = {
    fbcsp_aligned[arm_label].events[:, -1][0]: 0,
    fbcsp_aligned[leg_label].events[:, -1][0]: 1}
    labels_offline = np.vectorize(mapping_offline.get)(fbcsp_aligned.events[:, -1])
    return fbcsp_aligned, erp_aligned, labels_offline

In [19]:
def online_whole_data_classification(epochs_fbcsp, epochs_erp, labels,
                                     fbcsp_clf, fbcsp_csp, erp_clf, combined_clf,
                                     fbcsp_selected_features, erp_selected_features,
                                     fbcsp_scaler, erp_scaler):
    """
    Performs classification on the entire, unseen epochs and returns the accuracy.

    Args:
        epochs_fbcsp (mne.Epochs): FBCSP epochs (aligned and preprocessed).
        epochs_erp (mne.Epochs): ERP epochs (aligned and preprocessed).
        labels (np.ndarray): The true labels for the epochs.
        fbcsp_clf, fbcsp_csp, erp_clf, combined_clf: Pre-trained models.
        fbcsp_selected_features (np.ndarray): mRMR indices for FBCSP.
        erp_selected_features (np.ndarray): mRMR indices for ERP.
        fbcsp_scaler (StandardScaler): Pre-trained scaler for FBCSP features.
        erp_scaler (StandardScaler): Pre-trained scaler for ERP features.

    Returns:
        dict: A dictionary with the overall accuracy for each classification method.
    """

    # --- FBCSP Feature Extraction on the whole epoch ---
    fbcsp_features_list = []
    # FBCSP bands from your offline code
    filtered_bands = [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (13, 30)]

    for l_freq, h_freq in filtered_bands:
        temp_filtered_epochs = epochs_fbcsp.copy().filter(l_freq, h_freq, verbose=False)
        epochs_data = temp_filtered_epochs.get_data(copy=False) * 1e6
        csp_features = fbcsp_csp.transform(epochs_data)
        fbcsp_features_list.append(csp_features)

    final_fbcsp_features = np.concatenate(fbcsp_features_list, axis=1)

    # Apply pre-trained scaler before mRMR feature selection
    final_fbcsp_features = fbcsp_scaler.transform(final_fbcsp_features)

    # Apply mRMR feature selection with the pre-trained indices
    final_fbcsp_features = final_fbcsp_features[:, fbcsp_selected_features]

    # --- ERP Feature Extraction on the whole epoch with sliding window ---
    # The time window for online whole data classification must match offline training
    window_duration_s = 0.5
    step_size_s = 0.5
    sfreq = epochs_erp.info['sfreq']

    window_samples = int(window_duration_s * sfreq)
    step_samples = int(step_size_s * sfreq)

    # Define the time range for the sliding window to match offline training.
    start_time_s = 0.0
    end_time_s = 2.5
    
    # Correctly crop the online epochs to match the offline training window
    epochs_erp_cropped = epochs_erp.copy().crop(tmin=start_time_s, tmax=end_time_s, include_tmax=False)

    all_erp_features = []
    for epoch_data in epochs_erp_cropped.get_data():
        trial_features = []
        for start_idx in np.arange(0, epoch_data.shape[-1] - window_samples + 1, step_samples):
            end_idx = start_idx + window_samples
            window_data = epoch_data[:, start_idx:end_idx]
            window_features = np.mean(window_data, axis=1)
            trial_features.append(window_features)

        all_erp_features.append(np.concatenate(trial_features))

    final_erp_features = np.array(all_erp_features)

    # Apply pre-trained scaler before mRMR feature selection
    final_erp_features = erp_scaler.transform(final_erp_features)

    # Apply mRMR feature selection with the pre-trained indices
    final_erp_features = final_erp_features[:, erp_selected_features]

    # --- Classification and Accuracy Calculation ---

    fbcsp_preds = fbcsp_clf.predict(final_fbcsp_features)
    fbcsp_accuracy = accuracy_score(labels, fbcsp_preds)
    fbcsp_std = np.std(fbcsp_preds == labels)
    erp_preds = erp_clf.predict(final_erp_features)
    erp_accuracy = accuracy_score(labels, erp_preds)
    erp_std = np.std(erp_preds == labels)
    combined_features = np.concatenate([final_fbcsp_features, final_erp_features], axis=1)
    combined_preds = combined_clf.predict(combined_features)
    combined_accuracy = accuracy_score(labels, combined_preds)
    combined_std = np.std(combined_preds == labels)

    return {
        'fbcsp_results': [fbcsp_accuracy,fbcsp_std],
        'erp_results': [erp_accuracy,erp_std],
        'combined_results': [combined_accuracy,combined_std]
    }

In [20]:
def online_overtime_classification(epochs_fbcsp, epochs_erp, labels,
                                   models_and_selectors):
    """
    Performs classification over time using a sliding window on unseen data.

    Args:
        epochs_fbcsp (mne.Epochs): FBCSP epochs (aligned and preprocessed).
        epochs_erp (mne.Epochs): ERP epochs (aligned and preprocessed).
        labels (np.ndarray): The true labels for the epochs.
        models_and_selectors (dict): A dictionary containing all pre-trained models
                                     and selected feature indices.

    Returns:
        dict: A dictionary with accuracies for each classification method over time.
    """

    # Load pre-trained models and selectors from the dictionary
    fbcsp_clf = models_and_selectors['fbcsp_clf']
    fbcsp_csp = models_and_selectors['fbcsp_csp']
    fbcsp_selected_features = models_and_selectors['fbcsp_selected_features']
    fbcsp_scaler = models_and_selectors['fbcsp_scaler']
    erp_clf = models_and_selectors['erp_clf']
    erp_selected_features = models_and_selectors['erp_selected_features']
    erp_scaler = models_and_selectors['erp_scaler']
    combined_clf = models_and_selectors['combined_clf']

    all_fbcsp_preds = []
    all_erp_preds = []
    all_combined_preds = []

    # Define online window parameters. This is the main sliding window.
    window_duration_s = 2.5
    step_size_s = 0.25

    sfreq = epochs_fbcsp.info['sfreq']
    
    window_samples = int(window_duration_s * sfreq)
    step_samples = int(step_size_s * sfreq)

    # Determine the time range for the sliding window
    # The online epoch for FBCSP should go up to 5s, so we slide the 2.5s window over this period.
    start_time_online = 0.0
    end_time_online = 5.0
    
    try:
        start_sample_idx = epochs_fbcsp.time_as_index(start_time_online)[0]
        end_sample_idx_loop = epochs_fbcsp.time_as_index(end_time_online)[0]
    except IndexError:
        print("Warning: The online epochs do not contain the full time range for the sliding window. "
              "Please create the online epochs with a wider time window.")
        return None

    start_times = np.arange(start_sample_idx, end_sample_idx_loop - window_samples + 1, step_samples)
    
    # List to store the center time of each window
    window_labels = []

    for start_sample in start_times:
        end_sample = start_sample + window_samples

        # --- FBCSP Feature Extraction for the 2.5s window ---
        # The FBCSP offline training used a [0.5, 3.0] window, which has a duration of 2.5s.
        # We need to crop to that window. The start time of the online window is `epochs_fbcsp.times[start_sample]`.
        # So we crop from that start time to a duration of 2.5s.
        fbcsp_window_epochs = epochs_fbcsp.copy().crop(tmin=epochs_fbcsp.times[start_sample],
                                                       tmax=epochs_fbcsp.times[start_sample + window_samples - 1],
                                                       include_tmax=True)
        
        fbcsp_features_list = []
        filtered_bands = [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (13, 30)]
        
        for l_freq, h_freq in filtered_bands:
            temp_filtered_epochs = fbcsp_window_epochs.copy().filter(l_freq, h_freq, verbose=False)
            epochs_data = temp_filtered_epochs.get_data(copy=False) * 1e6
            csp_features = fbcsp_csp.transform(epochs_data)
            fbcsp_features_list.append(csp_features)
        
        fbcsp_features = np.concatenate(fbcsp_features_list, axis=1)
        
        # Apply pre-trained scaler before mRMR feature selection
        fbcsp_features = fbcsp_scaler.transform(fbcsp_features)
        
        # Apply mRMR feature selection with the pre-trained indices
        fbcsp_features_mrmr = fbcsp_features[:, fbcsp_selected_features]

        # --- ERP Feature Extraction for the 2.5s window with nested sliding windows ---
        # The ERP offline training used a [-0.5, 2.5] window. We need to crop the online data
        # to a 2.5s window starting from the same relative point.
        erp_window_epochs = epochs_erp.copy().crop(tmin=epochs_erp.times[start_sample],
                                                   tmax=epochs_erp.times[start_sample + window_samples - 1],
                                                   include_tmax=True)

        # Define nested ERP window parameters from offline code
        erp_window_duration_s = 0.5
        erp_step_size_s = 0.5
        erp_window_samples = int(erp_window_duration_s * sfreq)
        erp_step_samples = int(erp_step_size_s * sfreq)

        all_erp_features_window = []
        for trial_data in erp_window_epochs.get_data():
            trial_features = []
            for start_idx in np.arange(0, trial_data.shape[-1] - erp_window_samples + 1, erp_step_samples):
                end_idx = start_idx + erp_window_samples
                window_data = trial_data[:, start_idx:end_idx]
                window_features = np.mean(window_data, axis=1)
                trial_features.append(window_features)
            
            all_erp_features_window.append(np.concatenate(trial_features))

        erp_features = np.array(all_erp_features_window)
        
        # Apply pre-trained scaler before mRMR feature selection
        erp_features = erp_scaler.transform(erp_features)
        
        # Apply mRMR feature selection with the pre-trained indices
        erp_features_mrmr = erp_features[:, erp_selected_features]

        # --- Classification for all epochs in the current window ---
        fbcsp_preds = fbcsp_clf.predict(fbcsp_features_mrmr)
        erp_preds = erp_clf.predict(erp_features_mrmr)
        
        combined_features = np.concatenate([fbcsp_features_mrmr, erp_features_mrmr], axis=1)
        combined_preds = combined_clf.predict(combined_features)
        
        all_fbcsp_preds.append(fbcsp_preds)
        all_erp_preds.append(erp_preds)
        all_combined_preds.append(combined_preds)
        
        window_labels.append(epochs_fbcsp.times[start_sample + window_samples // 2])

    # Calculate the accuracy for each time window
    fbcsp_accuracies = [accuracy_score(labels, preds) for preds in all_fbcsp_preds]
    erp_accuracies = [accuracy_score(labels, preds) for preds in all_erp_preds]
    combined_accuracies = [accuracy_score(labels, preds) for preds in all_combined_preds]
    fbcsp_std = [np.std(preds == labels) for preds in all_fbcsp_preds]
    erp_std = [np.std(preds == labels) for preds in all_erp_preds]
    combined_std = [np.std(preds == labels) for preds in all_combined_preds]
    return {
        'times': window_labels,
        'fbcsp_results': [fbcsp_accuracies, fbcsp_std],
        'erp_results': [erp_accuracies, erp_std],
        'combined_results': [combined_accuracies, combined_std]
    }


In [21]:
classification_results= pd.read_excel('classification_results/offline_classification_results.xlsx')
classification_results

,Subject,Condition,Feature_Extraction,Accuracy,Selected_Features
0,1,AW,FBCSP,0.4125,"[6, 16, 10, 22, 19, 14, 12, 2, 20, 3]"
1,2,AW,FBCSP,0.5125,"[1, 6, 22, 4, 10, 16, 19, 12, 7, 14]"
2,3,AW,FBCSP,0.3750,"[18, 5, 13, 6, 19, 1, 15, 2, 11, 9]"
3,4,AW,FBCSP,0.5000,"[13, 3, 6, 2, 1, 18, 10, 5, 16, 9]"
4,5,AW,FBCSP,0.5375,"[18, 12, 6, 11, 8, 1, 16, 5, 20, 10]"
...,...,...,...,...,...
256,25,MI,Combined,0.7000,[]
257,26,MI,Combined,0.6875,[]
258,27,MI,Combined,0.6875,[]
259,28,MI,Combined,0.5375,[]


In [22]:
subjects = [str(i) for i in range(1,30)]

all_online_AW_results_seen = {}
all_online_AW_results_new = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')

    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    all_online_AW_results_seen[subject] = {}
    all_online_AW_results_new[subject] = {}

    AW_FBCSP_csp= joblib.load(os.path.join('features_extractors', 'AW_FBCSP_csp_subject_'+str(n)+'.pkl'))
    AW_FBCSP_scaler = joblib.load(os.path.join('features_extractors', 'AW_FBCSP_csp_scaler_subject_'+str(n)+'.pkl'))
    AW_FBCSP_clf = joblib.load(os.path.join('classifiers', 'AW_FBCSP_clf_subject_'+str(n)+'.pkl'))
    AW_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='AW')]['Selected_Features'].values[0])
    AW_ERP_clf = joblib.load(os.path.join('classifiers', 'AW_ERP_clf_subject_'+str(n)+'.pkl'))
    AW_ERP_scaler = joblib.load(os.path.join('features_extractors', 'AW_ERP_scaler_subject_'+str(n)+'.pkl'))
    AW_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='AW')]['Selected_Features'].values[0])
    AW_combined_clf = joblib.load(os.path.join('classifiers', 'AW_Combined_clf_subject_'+str(n)+'.pkl'))

    epochs_online_AW_FBCSP = create_online_epochs(online_file, 'AW','New_labels', FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    seen_epochs_FBCSP = epochs_online_AW_FBCSP['AW_Arm', 'AW_Leg']
    new_epochs_FBCSP = epochs_online_AW_FBCSP['AW_APLAUDIR', 'AW_PATEAR']

    epochs_online_AW_ERP = create_online_epochs(online_file, 'AW', 'New_labels', ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)
    seen_epochs_ERP = epochs_online_AW_ERP['AW_Arm', 'AW_Leg']
    new_epochs_ERP = epochs_online_AW_ERP['AW_APLAUDIR', 'AW_PATEAR']

    # Align epochs and labels
    
    AW_FBCSP_online_aligned_seen, AW_ERP_online_aligned_seen, labels_online_AW_seen = align_epochs_and_labels(seen_epochs_FBCSP, seen_epochs_ERP,'AW_Arm', 'AW_Leg')
    AW_FBCSP_online_aligned_new, AW_ERP_online_aligned_new, labels_online_AW_new = align_epochs_and_labels(new_epochs_FBCSP, new_epochs_ERP,'AW_APLAUDIR', 'AW_PATEAR')

    models_and_selectors = {'fbcsp_clf':AW_FBCSP_clf,'fbcsp_csp': AW_FBCSP_csp, 'fbcsp_selected_features': AW_FBCSP_feature_selected, 'fbcsp_scaler': AW_FBCSP_scaler,
                            'erp_clf': AW_ERP_clf, 'erp_selected_features': AW_ERP_feature_selected, 'erp_scaler': AW_ERP_scaler,
                            'combined_clf': AW_combined_clf}
    
    results = online_overtime_classification(AW_FBCSP_online_aligned_seen, AW_ERP_online_aligned_seen, labels_online_AW_seen, models_and_selectors) 
    all_online_AW_results_seen[subject] = results

    results = online_overtime_classification(AW_FBCSP_online_aligned_new, AW_ERP_online_aligned_new, labels_online_AW_new, models_and_selectors)
    all_online_AW_results_new[subject] = results

Online file found: raw_data\1\1_online.set
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Online file found: raw_data\4\4_online.set
Online file found: raw_data\5\5_online.set
Online file found: raw_data\6\6_online.set
Online file found: raw_data\7\7_online.set
Online file found: raw_data\8\8_online.set
Online file found: raw_data\9\9_online.set
Online file found: raw_data\10\10_online.set
Online file found: raw_data\11\11_online.set
Online file found: raw_data\12\12_online.set
Online file found: raw_data\13\13_online.set
Online file found: raw_data\14\14_online.set
Online file found: raw_data\15\15_online.set
Online file found: raw_data\16\16_online.set
Online file found: raw_data\17\17_online.set
Online file found: raw_data\18\18_online.set
Online file found: raw_data\19\19_online.set
Online file found: raw_data\20\20_online.set
Online file found: raw_data\21\21_online.set
Online file found: raw_data\22\22_online.set
Online file not found: r

In [23]:
all_online_MI_results = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')

    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    all_online_MI_results[subject] = {}

    MI_FBCSP_csp = joblib.load(os.path.join('features_extractors', f'MI_FBCSP_csp_subject_{n}.pkl'))
    MI_FBCSP_scaler = joblib.load(os.path.join('features_extractors', f'MI_FBCSP_csp_scaler_subject_{n}.pkl'))
    MI_FBCSP_clf = joblib.load(os.path.join('classifiers', f'MI_FBCSP_clf_subject_{n}.pkl'))
    MI_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MI')]['Selected_Features'].values[0])    
    MI_ERP_clf = joblib.load(os.path.join('classifiers', f'MI_ERP_clf_subject_{n}.pkl'))
    MI_ERP_scaler = joblib.load(os.path.join('features_extractors', f'MI_ERP_scaler_subject_{n}.pkl'))
    MI_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MI')]['Selected_Features'].values[0])    
    MI_combined_clf = joblib.load(os.path.join('classifiers', f'MI_Combined_clf_subject_{n}.pkl'))
    epochs_online_MI_FBCSP = create_online_epochs(online_file, 'MI', None, FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    epochs_online_MI_ERP = create_online_epochs(online_file, 'MI', None, ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)

    MI_FBCSP_online_aligned, MI_ERP_online_aligned, labels_online_MI = align_epochs_and_labels(
        epochs_online_MI_FBCSP, epochs_online_MI_ERP, 'MI_Arm', 'MI_Leg'
    )

    models_and_selectors = {
        'fbcsp_clf': MI_FBCSP_clf,
        'fbcsp_csp': MI_FBCSP_csp,
        'fbcsp_selected_features': MI_FBCSP_feature_selected,
        'fbcsp_scaler': MI_FBCSP_scaler,
        'erp_clf': MI_ERP_clf,
        'erp_selected_features': MI_ERP_feature_selected,
        'erp_scaler': MI_ERP_scaler,
        'combined_clf': MI_combined_clf
    }

    results = online_overtime_classification(
        MI_FBCSP_online_aligned, MI_ERP_online_aligned, labels_online_MI, models_and_selectors
    )
    all_online_MI_results[subject] = results

Online file found: raw_data\1\1_online.set
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Online file found: raw_data\4\4_online.set
Online file found: raw_data\5\5_online.set
Online file found: raw_data\6\6_online.set
Online file found: raw_data\7\7_online.set
Online file found: raw_data\8\8_online.set
Online file found: raw_data\9\9_online.set
Online file found: raw_data\10\10_online.set
Online file found: raw_data\11\11_online.set
Online file found: raw_data\12\12_online.set
Online file found: raw_data\13\13_online.set
Online file found: raw_data\14\14_online.set
Online file found: raw_data\15\15_online.set
Online file found: raw_data\16\16_online.set
Online file found: raw_data\17\17_online.set
Online file found: raw_data\18\18_online.set
Online file found: raw_data\19\19_online.set
Online file found: raw_data\20\20_online.set
Online file found: raw_data\21\21_online.set
Online file found: raw_data\22\22_online.set
Online file not found: r

In [24]:
all_online_MO_results = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')

    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    all_online_MO_results[subject] = {}

    MO_FBCSP_csp = joblib.load(os.path.join('features_extractors', f'MO_FBCSP_csp_subject_{n}.pkl'))
    MO_FBCSP_scaler = joblib.load(os.path.join('features_extractors', f'MO_FBCSP_csp_scaler_subject_{n}.pkl'))
    MO_FBCSP_clf = joblib.load(os.path.join('classifiers', f'MO_FBCSP_clf_subject_{n}.pkl'))
    MO_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MO')]['Selected_Features'].values[0])    
    MO_ERP_clf = joblib.load(os.path.join('classifiers', f'MO_ERP_clf_subject_{n}.pkl'))
    MO_ERP_scaler = joblib.load(os.path.join('features_extractors', f'MO_ERP_scaler_subject_{n}.pkl'))
    MO_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MO')]['Selected_Features'].values[0])
    MO_combined_clf = joblib.load(os.path.join('classifiers', f'MO_Combined_clf_subject_{n}.pkl'))
    epochs_online_MO_FBCSP = create_online_epochs(online_file, 'MO', None, FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    epochs_online_MO_ERP = create_online_epochs(online_file, 'MO', None, ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)

    MO_FBCSP_online_aligned, MO_ERP_online_aligned, labels_online_MO = align_epochs_and_labels(
        epochs_online_MO_FBCSP, epochs_online_MO_ERP, 'MO_Arm', 'MO_Leg'
    )

    models_and_selectors = {
        'fbcsp_clf': MO_FBCSP_clf,
        'fbcsp_csp': MO_FBCSP_csp,
        'fbcsp_selected_features': MO_FBCSP_feature_selected,
        'fbcsp_scaler': MO_FBCSP_scaler,
        'erp_clf': MO_ERP_clf,
        'erp_selected_features': MO_ERP_feature_selected,
        'erp_scaler': MO_ERP_scaler,
        'combined_clf': MO_combined_clf
    }

    results = online_overtime_classification(
        MO_FBCSP_online_aligned, MO_ERP_online_aligned, labels_online_MO, models_and_selectors
    )
    all_online_MO_results[subject] = results

Online file found: raw_data\1\1_online.set
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Online file found: raw_data\4\4_online.set
Online file found: raw_data\5\5_online.set
Online file found: raw_data\6\6_online.set
Online file found: raw_data\7\7_online.set
Online file found: raw_data\8\8_online.set
Online file found: raw_data\9\9_online.set
Online file found: raw_data\10\10_online.set
Online file found: raw_data\11\11_online.set
Online file found: raw_data\12\12_online.set
Online file found: raw_data\13\13_online.set
Online file found: raw_data\14\14_online.set
Online file found: raw_data\15\15_online.set
Online file found: raw_data\16\16_online.set
Online file found: raw_data\17\17_online.set
Online file found: raw_data\18\18_online.set
Online file found: raw_data\19\19_online.set
Online file found: raw_data\20\20_online.set
Online file found: raw_data\21\21_online.set
Online file found: raw_data\22\22_online.set
Online file not found: r

In [56]:
data = []
for subject_id, scores in all_online_AW_results_new.items():
    for j, time in enumerate(scores['times']):
        data.append({
            'Subject': subject_id,
            'times': time,
            'Condition': 'AW_New',
            'FBCSP_acc': scores['fbcsp_results'][0][j],
            'FBCSP_std': scores['fbcsp_results'][1][j],
            'ERP_acc': scores['erp_results'][0][j],
            'ERP_std': scores['erp_results'][1][j],
            'Combined_acc': scores['combined_results'][0][j],
            'Combined_std': scores['combined_results'][1][j]
        })
for subject_id, scores in all_online_AW_results_seen.items():
    for j, time in enumerate(scores['times']):
        data.append({
            'Subject': subject_id,
            'times': time,
            'Condition': 'AW_Seen',
            'FBCSP_acc': scores['fbcsp_results'][0][j],
            'FBCSP_std': scores['fbcsp_results'][1][j],
            'ERP_acc': scores['erp_results'][0][j],
            'ERP_std': scores['erp_results'][1][j],
            'Combined_acc': scores['combined_results'][0][j],
            'Combined_std': scores['combined_results'][1][j]
        })
for subject_id, scores in all_online_MI_results.items():
    for j, time in enumerate(scores['times']):
        data.append({
            'Subject': subject_id,
            'times': time,
            'Condition': 'MI',
            'FBCSP_acc': scores['fbcsp_results'][0][j],
            'FBCSP_std': scores['fbcsp_results'][1][j],
            'ERP_acc': scores['erp_results'][0][j],
            'ERP_std': scores['erp_results'][1][j],
            'Combined_acc': scores['combined_results'][0][j],
            'Combined_std': scores['combined_results'][1][j]
        })
for subject_id, scores in all_online_MO_results.items():
    for j, time in enumerate(scores['times']):
        data.append({
            'Subject': subject_id,
            'times': time,
            'Condition': 'MO',
            'FBCSP_acc': scores['fbcsp_results'][0][j],
            'FBCSP_std': scores['fbcsp_results'][1][j],
            'ERP_acc': scores['erp_results'][0][j],
            'ERP_std': scores['erp_results'][1][j],
            'Combined_acc': scores['combined_results'][0][j],
            'Combined_std': scores['combined_results'][1][j]
        })

df=pd.DataFrame(data)
df.to_excel('classification_results/online_overtime_classification_results.xlsx', index=False)
df.groupby(['Condition']).agg(['mean', 'max', 'min']).drop(['times','Subject'], axis=1)

FBCSP_acc              FBCSP_std                  ERP_acc            \
               mean   max    min      mean  max       min      mean       max   
Condition                                                                       
AW_New     0.494242  0.75  0.250  0.494735  0.5  0.433013  0.532121  0.916667   
AW_Seen    0.491364  0.75  0.125  0.488963  0.5  0.330719  0.539545  1.000000   
MI         0.512000  0.70  0.350  0.496127  0.5  0.458258  0.504364  0.800000   
MO         0.511273  0.70  0.350  0.496692  0.5  0.458258  0.496545  0.750000   

                  ERP_std                Combined_acc                      \
            min      mean  max       min         mean       max       min   
Condition                                                                   
AW_New     0.00  0.476274  0.5  0.000000     0.496970  0.916667  0.166667   
AW_Seen    0.00  0.464721  0.5  0.000000     0.492273  1.000000  0.000000   
MI         0.25  0.489033  0.5  0.400000     0.502909  0.750000  0.350000   
MO         0.15  0.487317  0.5  0.357071     0.510545  0.700000  0.350000   

          Combined_std                 
                  mean  max       min  
Condition                              
AW_New        0.489923  0.5  0.276385  
AW_Seen       0.470440  0.5  0.000000  
MI            0.496005  0.5  0.433013  
MO            0.495957  0.5  0.458258

In [ ]:

whole_epochs_results_MI = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')
    
    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    whole_epochs_results_MI[subject] = {}

    # Load pre-trained models
    fbcsp_clf = joblib.load(os.path.join('classifiers', f'MI_FBCSP_clf_subject_{n}.pkl'))
    fbcsp_csp = joblib.load(os.path.join('features_extractors', f'MI_FBCSP_csp_subject_{n}.pkl'))
    fbcsp_scaler = joblib.load(os.path.join('features_extractors', f'MI_FBCSP_csp_scaler_subject_{n}.pkl'))
    MI_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MI')]['Selected_Features'].values[0])    
    erp_clf = joblib.load(os.path.join('classifiers', f'MI_ERP_clf_subject_{n}.pkl'))
    erp_scaler = joblib.load(os.path.join('features_extractors', f'MI_ERP_scaler_subject_{n}.pkl'))
    MI_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MI')]['Selected_Features'].values[0])    

    combined_clf = joblib.load(os.path.join('classifiers', f'MI_Combined_clf_subject_{n}.pkl'))

    # Create aligned epochs
    epochs_online_MI_FBCSP = create_online_epochs(online_file, 'MI', None, FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    epochs_online_MI_ERP = create_online_epochs(online_file, 'MI', None, ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)

    MI_FBCSP_online_aligned, MI_ERP_online_aligned, labels_online_MI = align_epochs_and_labels(
        epochs_online_MI_FBCSP, epochs_online_MI_ERP, 'MI_Arm', 'MI_Leg'
    )

    # Run the whole-data classification method
    whole_data_results = online_whole_data_classification(
        MI_FBCSP_online_aligned, MI_ERP_online_aligned, labels_online_MI,
        fbcsp_clf, fbcsp_csp, erp_clf, combined_clf, MI_FBCSP_feature_selected,
        MI_ERP_feature_selected,
        fbcsp_scaler, erp_scaler
    )
    whole_epochs_results_MI[subject] = whole_data_results
    print(f"Whole-data classification results for Subject {subject}:")
    print(whole_data_results)

Online file found: raw_data\1\1_online.set
Whole-data classification results for Subject 1:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.55, 0.49749371855331], 'combined_results': [0.45, 0.49749371855331]}
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Whole-data classification results for Subject 3:
{'fbcsp_results': [0.65, 0.47696960070847283], 'erp_results': [0.5, 0.5], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\4\4_online.set
Whole-data classification results for Subject 4:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.55, 0.49749371855331], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\5\5_online.set
Whole-data classification results for Subject 5:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.55, 0.4974937185533099], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\6\6_online.set
Whole-data classification results for Subject 6:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.35, 0.47696960

In [27]:
whole_epochs_results_MO = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')
    
    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    whole_epochs_results_MO[subject] = {}

    # Load pre-trained models
    fbcsp_clf = joblib.load(os.path.join('classifiers', f'MO_FBCSP_clf_subject_{n}.pkl'))
    fbcsp_csp = joblib.load(os.path.join('features_extractors', f'MO_FBCSP_csp_subject_{n}.pkl'))
    fbcsp_scaler = joblib.load(os.path.join('features_extractors', f'MO_FBCSP_csp_scaler_subject_{n}.pkl'))
    MO_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MO')]['Selected_Features'].values[0])    

    erp_clf = joblib.load(os.path.join('classifiers', f'MO_ERP_clf_subject_{n}.pkl'))
    erp_scaler = joblib.load(os.path.join('features_extractors', f'MO_ERP_scaler_subject_{n}.pkl'))
    MO_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='MO')]['Selected_Features'].values[0])    

    combined_clf = joblib.load(os.path.join('classifiers', f'MO_Combined_clf_subject_{n}.pkl'))

    # Create aligned epochs
    epochs_online_MO_FBCSP = create_online_epochs(online_file, 'MO', None, FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    epochs_online_MO_ERP = create_online_epochs(online_file, 'MO', None, ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)

    MO_FBCSP_online_aligned, MO_ERP_online_aligned, labels_online_MO = align_epochs_and_labels(
        epochs_online_MO_FBCSP, epochs_online_MO_ERP, 'MO_Arm', 'MO_Leg'
    )

    # Run the whole-data classification method
    whole_data_results = online_whole_data_classification(
        MO_FBCSP_online_aligned, MO_ERP_online_aligned, labels_online_MO,
        fbcsp_clf, fbcsp_csp, erp_clf, combined_clf, MO_FBCSP_feature_selected,
        MO_ERP_feature_selected,
        fbcsp_scaler, erp_scaler
    )
    whole_epochs_results_MO[subject] = whole_data_results
    print(f"Whole-data classification results for Subject {subject}:")
    print(whole_data_results)

Online file found: raw_data\1\1_online.set
Whole-data classification results for Subject 1:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.55, 0.49749371855331], 'combined_results': [0.5, 0.5]}
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Whole-data classification results for Subject 3:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.45, 0.49749371855331004], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\4\4_online.set
Whole-data classification results for Subject 4:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.45, 0.49749371855331004], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\5\5_online.set
Whole-data classification results for Subject 5:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.55, 0.4974937185533099], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\6\6_online.set
Whole-data classification results for Subject 6:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.7, 0.45825756949558394], 

In [28]:
whole_epochs_results_AW_seen = {}
whole_epochs_results_AW_new = {}
for n, subject in enumerate(subjects):
    #online_file = os.path.join(data_folder, str(subject), f'{subject}_cleaned_online.set')
    online_file = os.path.join(data_folder, str(subject), f'{subject}_online.set')

    if os.path.exists(online_file):
        print(f"Online file found: {online_file}")
    else:
        print(f"Online file not found: {online_file}")
        continue
    whole_epochs_results_AW_seen[subject] = {}
    whole_epochs_results_AW_new[subject] = {}

    AW_FBCSP_csp= joblib.load(os.path.join('features_extractors', 'AW_FBCSP_csp_subject_'+str(n)+'.pkl'))
    AW_FBCSP_scaler = joblib.load(os.path.join('features_extractors', 'AW_FBCSP_csp_scaler_subject_'+str(n)+'.pkl'))
    AW_FBCSP_clf = joblib.load(os.path.join('classifiers', 'AW_FBCSP_clf_subject_'+str(n)+'.pkl'))
    AW_FBCSP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='FBCSP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='AW')]['Selected_Features'].values[0])

    AW_ERP_clf = joblib.load(os.path.join('classifiers', 'AW_ERP_clf_subject_'+str(n)+'.pkl'))
    AW_ERP_scaler = joblib.load(os.path.join('features_extractors', 'AW_ERP_scaler_subject_'+str(n)+'.pkl'))
    AW_ERP_feature_selected = ast.literal_eval(classification_results[(classification_results['Feature_Extraction']=='ERP') & (classification_results['Subject']==int(subject)) & (classification_results['Condition']=='AW')]['Selected_Features'].values[0])

    AW_combined_clf = joblib.load(os.path.join('classifiers', 'AW_Combined_clf_subject_'+str(n)+'.pkl'))

    epochs_online_AW_FBCSP = create_online_epochs(online_file, 'AW','New_labels', FBCSP_channels, [-0.5, 5.0], None)#, clean_data=False)
    seen_epochs_FBCSP = epochs_online_AW_FBCSP['AW_Arm', 'AW_Leg']
    new_epochs_FBCSP = epochs_online_AW_FBCSP['AW_APLAUDIR', 'AW_PATEAR']

    epochs_online_AW_ERP = create_online_epochs(online_file, 'AW', 'New_labels', ERP_channels, [-0.5, 5.0], [-0.5, 0])#, clean_data=False)
    seen_epochs_ERP = epochs_online_AW_ERP['AW_Arm', 'AW_Leg']
    new_epochs_ERP = epochs_online_AW_ERP['AW_APLAUDIR', 'AW_PATEAR']

    # Align epochs and labels
    
    AW_FBCSP_online_aligned_seen, AW_ERP_online_aligned_seen, labels_online_AW_seen = align_epochs_and_labels(seen_epochs_FBCSP, seen_epochs_ERP,'AW_Arm', 'AW_Leg')
    AW_FBCSP_online_aligned_new, AW_ERP_online_aligned_new, labels_online_AW_new = align_epochs_and_labels(new_epochs_FBCSP, new_epochs_ERP,'AW_APLAUDIR', 'AW_PATEAR')
   
   # Run the whole-data classification method
    whole_data_results = online_whole_data_classification(
        AW_FBCSP_online_aligned_seen, AW_ERP_online_aligned_seen, labels_online_AW_seen,
        AW_FBCSP_clf, AW_FBCSP_csp, AW_ERP_clf, AW_combined_clf, AW_FBCSP_feature_selected,
        AW_ERP_feature_selected,
        AW_FBCSP_scaler, AW_ERP_scaler
    )
    whole_epochs_results_AW_seen[subject] = whole_data_results
    print(f"Seen Whole-data classification results for Subject {subject}:")
    print(whole_data_results)

    whole_data_results = online_whole_data_classification(
        AW_FBCSP_online_aligned_new, AW_ERP_online_aligned_new, labels_online_AW_new,
        AW_FBCSP_clf, AW_FBCSP_csp, AW_ERP_clf, AW_combined_clf, AW_FBCSP_feature_selected,
        AW_ERP_feature_selected,
        AW_FBCSP_scaler, AW_ERP_scaler
    )
    whole_epochs_results_AW_new[subject] = whole_data_results
    print(f"New Whole-data classification results for Subject {subject}:")
    print(whole_data_results)


Online file found: raw_data\1\1_online.set
Seen Whole-data classification results for Subject 1:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.75, 0.4330127018922193], 'combined_results': [0.5, 0.5]}
New Whole-data classification results for Subject 1:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.5833333333333334, 0.4930066485916347], 'combined_results': [0.5, 0.5]}
Online file not found: raw_data\2\2_online.set
Online file found: raw_data\3\3_online.set
Seen Whole-data classification results for Subject 3:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.875, 0.33071891388307384], 'combined_results': [0.5, 0.5]}
New Whole-data classification results for Subject 3:
{'fbcsp_results': [0.5, 0.5], 'erp_results': [0.5, 0.5], 'combined_results': [0.5, 0.5]}
Online file found: raw_data\4\4_online.set
Seen Whole-data classification results for Subject 4:
{'fbcsp_results': [0.625, 0.4841229182759271], 'erp_results': [0.625, 0.4841229182759271], 'combined_results': [0.5, 0.5]}
New Whole-dat

In [55]:
# Create a DataFrame to organize the data
data = []
for subject_id, scores in whole_epochs_results_AW_new.items():
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_New',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['combined_results'][0],
        'Std': scores['combined_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_New',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['fbcsp_results'][0],
        'Std': scores['fbcsp_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_New',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['erp_results'][0],
        'Std': scores['erp_results'][1]
    })
for subject_id, scores in whole_epochs_results_AW_seen.items():
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_Seen',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['combined_results'][0],
        'Std': scores['combined_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_Seen',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['fbcsp_results'][0],
        'Std': scores['fbcsp_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'AW_Seen',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['erp_results'][0],
        'Std': scores['erp_results'][1]
    })

for subject_id, scores in whole_epochs_results_MI.items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['combined_results'][0],
        'Std': scores['combined_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['fbcsp_results'][0],
        'Std': scores['fbcsp_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['erp_results'][0],
        'Std': scores['erp_results'][1]
    })
for subject_id, scores in whole_epochs_results_MO.items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['combined_results'][0],
        'Std': scores['combined_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['fbcsp_results'][0],
        'Std': scores['fbcsp_results'][1]
    })
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['erp_results'][0],
        'Std': scores['erp_results'][1]
    })


df = pd.DataFrame(data)
df.to_excel('classification_results/whole_epochs_classification_results.xlsx', index=False)
df.groupby(['Condition']).agg(['mean', 'max', 'min']).drop(['Subject'], axis=1)

Accuracy                        Std               
               mean    max       min      mean  max       min
Condition                                                    
AW_New     0.508889  0.750  0.166667  0.485995  0.5  0.372678
AW_Seen    0.518333  0.875  0.125000  0.469765  0.5  0.330719
MI         0.510667  0.750  0.350000  0.494379  0.5  0.433013
MO         0.514667  0.700  0.300000  0.494818  0.5  0.458258